In [1]:
# import necessary libraries

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

import logging
logging.basicConfig(level = logging.INFO,
                    filename = 'model.log',
                    filemode = 'w',
                    format = '%(asctime)s - %(message)s - %(levelname)s',
                    force = True)


In [2]:
df = pd.read_csv(r'C:\MLOPs_HousePrice_PredictionModel\data\raw\house_price_prediction_dataset.csv')
df
    

,Property_ID,Date_Sold,Property_Type,Location_Type,Neighborhood_Quality,Condition,SqFt,Lot_Size_SqFt,Bedrooms,Bathrooms,...,Has_Pool,Has_Basement,Has_Fireplace,Has_Central_AC,Heating_Type,Distance_To_City_Center_Miles,School_Rating,HOA_Fee_Monthly,Annual_Property_Tax,Price
0,PROP_00001,2024-11-12,Single Family,Suburban,Low,Good,4767,16551,6,5.0,...,1,0,1,1,Gas,32.52,10,0,11506.58,1216500.0
1,PROP_00002,2025-10-11,Condo,Rural,Low,Good,838,13312,2,1.5,...,1,1,1,0,Gas,32.64,5,564,3547.88,435500.0
2,PROP_00003,2024-02-20,Condo,Rural,Low,Good,2315,12290,4,3.5,...,0,0,0,1,Electric,34.77,4,283,9858.71,709600.0
3,PROP_00004,2025-04-29,Condo,Rural,Low,Excellent,2204,10977,3,2.5,...,0,1,0,1,Electric,33.50,2,228,7886.23,559400.0
4,PROP_00005,2024-11-16,Condo,Suburban,Low,Good,4281,13885,6,5.0,...,0,0,1,1,Gas,8.34,1,410,10687.16,1250800.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34562,PROP_34563,2024-08-13,Townhouse,Urban,Low,Excellent,987,17359,2,2.0,...,1,0,0,1,Gas,6.02,3,510,5896.38,431900.0
34563,PROP_34564,2023-01-31,Single Family,Urban,Medium,Fair,2730,11634,4,3.0,...,0,1,1,1,Gas,30.85,2,100,8432.07,611700.0
34564,PROP_34565,2026-03-18,Single Family,Urban,Low,Good,3411,6192,6,5.0,...,0,1,0,1,Gas,2.00,4,0,10488.26,950700.0
34565,PROP_34566,2023-05-14,Townhouse,Suburban,Medium,Good,2768,17573,5,5.0,...,0,1,0,1,Electric,24.03,4,274,8324.61,803100.0


In [3]:
df.info()
df.describe()
df.describe(include='object')

<class 'pandas.DataFrame'>
RangeIndex: 34567 entries, 0 to 34566
Data columns (total 23 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Property_ID                    34567 non-null  str    
 1   Date_Sold                      34567 non-null  str    
 2   Property_Type                  34567 non-null  str    
 3   Location_Type                  34567 non-null  str    
 4   Neighborhood_Quality           34567 non-null  str    
 5   Condition                      34567 non-null  str    
 6   SqFt                           34567 non-null  int64  
 7   Lot_Size_SqFt                  34567 non-null  int64  
 8   Bedrooms                       34567 non-null  int64  
 9   Bathrooms                      34567 non-null  float64
 10  Floors                         34567 non-null  int64  
 11  Year_Built                     34567 non-null  int64  
 12  Garage_Spaces                  34567 non-null  int64  
 1

,Property_ID,Date_Sold,Property_Type,Location_Type,Neighborhood_Quality,Condition,Heating_Type
count,34567,34567,34567,34567,34567,34567,34567
unique,34567,1826,4,3,4,4,4
top,PROP_00001,2024-06-26,Single Family,Suburban,Medium,Good,Gas
freq,1,33,18990,17216,17304,17341,20596


In [4]:
df.isnull().sum()

Property_ID                      0
Date_Sold                        0
Property_Type                    0
Location_Type                    0
Neighborhood_Quality             0
Condition                        0
SqFt                             0
Lot_Size_SqFt                    0
Bedrooms                         0
Bathrooms                        0
Floors                           0
Year_Built                       0
Garage_Spaces                    0
Has_Pool                         0
Has_Basement                     0
Has_Fireplace                    0
Has_Central_AC                   0
Heating_Type                     0
Distance_To_City_Center_Miles    0
School_Rating                    0
HOA_Fee_Monthly                  0
Annual_Property_Tax              0
Price                            0
dtype: int64

In [5]:
from collections import OrderedDict

numerical_columns = df.select_dtypes(exclude='object').columns
categorical_col = df.select_dtypes(include = 'object').columns

stats = []

for i in numerical_columns:
    Q1 = df[i].quantile(0.25)
    Q2 = df[i].quantile(0.50)
    Q3 = df[i].quantile(0.75)

    IQR = Q3 - Q1

    LW = Q1 - 1.5 * IQR
    UW = Q3 + 1.5 * IQR

    numerical_stats = OrderedDict({
        'features': i,
        'minimum': df[i].min(),
        'maximum': df[i].max(),
        'mean': df[i].mean(),
        'median': df[i].median(),
        'standard deviation': df[i].std(),
        'variance': df[i].var(),
        'skewness': df[i].skew(),
        'kurtosis': df[i].kurt(),
        'Q1': Q1,
        'Q2': Q2,
        'Q3': Q3,
        'IQR': IQR,
        'LW': LW,
        'UW': UW,
        'Outliers': df[(df[i] < LW) | (df[i] > UW)].shape[0]
    })

    stats.append(numerical_stats)
print(stats)

[OrderedDict({'features': 'SqFt', 'minimum': np.int64(600), 'maximum': np.int64(4999), 'mean': np.float64(2805.38438395001), 'median': np.float64(2801.0), 'standard deviation': np.float64(1270.122418298215), 'variance': np.float64(1613210.957463706), 'skewness': np.float64(-0.00403108166585047), 'kurtosis': np.float64(-1.2019565827871939), 'Q1': np.float64(1711.0), 'Q2': np.float64(2801.0), 'Q3': np.float64(3910.0), 'IQR': np.float64(2199.0), 'LW': np.float64(-1587.5), 'UW': np.float64(7208.5), 'Outliers': 0}), OrderedDict({'features': 'Lot_Size_SqFt', 'minimum': np.int64(1665), 'maximum': np.int64(24912), 'mean': np.float64(13256.386929730668), 'median': np.float64(13245.0), 'standard deviation': np.float64(5619.124577865892), 'variance': np.float64(31574561.021576546), 'skewness': np.float64(0.01121123976123134), 'kurtosis': np.float64(-1.0789446626060066), 'Q1': np.float64(8485.5), 'Q2': np.float64(13245.0), 'Q3': np.float64(17978.0), 'IQR': np.float64(9492.5), 'LW': np.float64(-575